# Marketplace uplift x auction: walkthrough

This notebook reproduces the headline numbers of the pipeline end to end at small scale (~5 minutes):

1. Generate simulated marketplace data (`--scale small`, 50k impressions)
2. Build features and train the CTR ranker
3. Score the splits (`pctr`)
4. Fit + evaluate the causal estimators against ground truth
5. Simulate the ads auction
6. Run the mechanism x targeting experiment grid and write `reports/REPORT.md`

Everything is deterministic (fixed seeds); results land in `reports/` and `artifacts/`.

## 1. Data: generate + features

In [1]:
!uv run python -m mua.cli generate --scale small

2026-09-26 18:11:48,941 INFO    mua: Small scale: 50000 rows


2026-09-26 18:11:48,942 INFO    mua.sim.generate: Generating consumers ...
2026-09-26 18:11:48,971 INFO    mua.sim.generate: Generating merchants ...
2026-09-26 18:11:48,984 INFO    mua.sim.generate: Generating dashers ...
2026-09-26 18:11:48,985 INFO    mua.sim.generate: Building 50000 impressions ...


2026-09-26 18:11:49,114 INFO    mua.sim.generate: Writing Parquet datasets ...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties


26/09/26 18:11:53 WARN Utils: Your hostname, Abhirams-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.123 instead (on interface en0)
26/09/26 18:11:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/26 18:11:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/26 18:12:28 WARN TaskSetManager: Stage 3 contains a task of very large size (2726 KiB). The maximum recommended task size is 1000 KiB.


2026-09-26 18:12:39,183 INFO    mua.sim.generate: Generation complete. Stats:
2026-09-26 18:12:39,184 INFO    mua.sim.generate:   consumers=50000 merchants=5000 dashers=800
2026-09-26 18:12:39,184 INFO    mua.sim.generate:   impressions=50000 days=30
2026-09-26 18:12:39,184 INFO    mua.sim.generate:   treated share      = 0.3520
2026-09-26 18:12:39,184 INFO    mua.sim.generate:   base order rate    = 0.0787
2026-09-26 18:12:39,184 INFO    mua.sim.generate:   mean true_tau      = 0.0240
2026-09-26 18:12:39,184 INFO    mua.sim.generate:   mean true propensity = 0.3500
2026-09-26 18:12:39,184 INFO    mua.sim.generate:   mean baseline prob   = 0.0673
2026-09-26 18:12:39,184 INFO    mua.sim.generate:   naive ATE (diff in means) = 0.0321
2026-09-26 18:12:39,185 INFO    py4j.clientserver: Closing down clientserver connection


In [2]:
!uv run python -m mua.cli features

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/26 18:12:42 WARN Utils: Your hostname, Abhirams-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.123 instead (on interface en0)


26/09/26 18:12:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/26 18:12:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/26 18:13:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


2026-09-26 18:13:20,855 INFO    mua.features.build: Wrote train (35006 rows) -> /Users/abhiramkasireddi/MarketplaceUpliftAuction/data/processed/train


2026-09-26 18:13:25,328 INFO    mua.features.build: Wrote valid (6664 rows) -> /Users/abhiramkasireddi/MarketplaceUpliftAuction/data/processed/valid


2026-09-26 18:13:28,156 INFO    mua.features.build: Wrote test (8330 rows) -> /Users/abhiramkasireddi/MarketplaceUpliftAuction/data/processed/test


2026-09-26 18:13:29,107 INFO    mua.features.build: Total joined rows: 50000
2026-09-26 18:13:29,108 INFO    mua.features.build: Cuisine one-hot columns: ['cuisine_american', 'cuisine_burgers', 'cuisine_chinese', 'cuisine_dessert', 'cuisine_indian', 'cuisine_italian', 'cuisine_korean', 'cuisine_mexican', 'cuisine_pizza', 'cuisine_sushi', 'cuisine_thai', 'cuisine_vietnamese']


2026-09-26 18:13:29,177 INFO    mua.features.build: Feature columns (43): ['merchant_id', 'consumer_id', 'impression_id', 'hour_of_day', 'day_of_week', 'is_peak', 'slate_rank', 'consumer_merchant_distance_km', 'current_dasher_supply_ratio', 'estimated_delivery_time_min', 'promo_treated', 'clicked', 'ordered', 'order_value_usd', 'delivery_fee_usd', 'promo_cost_usd', 'day', 'price_sensitivity', 'order_frequency_prior', 'distance_tolerance_km', 'is_new_user', 'tenure_days', 'consumer_x_km', 'consumer_y_km', 'avg_rating', 'avg_prep_time_min', 'price_tier', 'historical_ctr', 'is_ads_advertiser', 'merchant_x_km', 'merchant_y_km', 'cuisine_american', 'cuisine_burgers', 'cuisine_chinese', 'cuisine_dessert', 'cuisine_indian', 'cuisine_italian', 'cuisine_korean', 'cuisine_mexican', 'cuisine_pizza', 'cuisine_sushi', 'cuisine_thai', 'cuisine_vietnamese']
2026-09-26 18:13:29,181 INFO    py4j.clientserver: Closing down clientserver connection


## 2. CTR ranker: train + score

(Fewer epochs on the small dataset; the quality gates are tuned for the full 2M-row run.)

In [3]:
!uv run python -m mua.cli train-ranker --epochs 4

2026-09-26 18:13:49,077 INFO    mua.ranker.train: Feature spec saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/artifacts/ranker/feature_spec.json


2026-09-26 18:13:58,827 INFO    mua.ranker.train: epoch  1/4: train_loss=0.71956 val_loss=0.69828 val_auc=0.5762 val_prauc=0.0784 lr=0.000894


2026-09-26 18:14:01,158 INFO    mua.ranker.train: epoch  2/4: train_loss=0.62998 val_loss=0.66416 val_auc=0.6187 val_prauc=0.0918 lr=0.000851


2026-09-26 18:14:03,966 INFO    mua.ranker.train: epoch  3/4: train_loss=0.54293 val_loss=0.61060 val_auc=0.6330 val_prauc=0.0944 lr=0.000443


2026-09-26 18:14:06,133 INFO    mua.ranker.train: epoch  4/4: train_loss=0.49961 val_loss=0.57073 val_auc=0.6375 val_prauc=0.0954 lr=0.000101


2026-09-26 18:14:06,200 INFO    mua.ranker.train: Best checkpoint saved (epoch 4) -> /Users/abhiramkasireddi/MarketplaceUpliftAuction/artifacts/ranker/best_model.pt


2026-09-26 18:14:07,500 INFO    mua.ranker.calibrate: Reliability diagram saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/calibration.png


2026-09-26 18:14:07,667 INFO    mua.ranker.calibrate: Calibration: valid ECE 0.3656 -> 0.0000 | test ECE 0.3658 -> 0.0047 | test AUC 0.6312


2026-09-26 18:14:07,864 INFO    mua.ranker.train: Run summary:
2026-09-26 18:14:07,865 INFO    mua.ranker.train:   training: best_epoch=4 best_val_logloss=0.57073 val_roc_auc=0.6375 val_pr_auc=0.0954 elapsed=9.8s
2026-09-26 18:14:07,865 INFO    mua.ranker.train:   test: roc_auc=0.6312 pr_auc=0.0993 | ECE before=0.3658 after=0.0047
2026-09-26 18:14:07,865 INFO    mua.ranker.train:   valid: ECE before=0.3656 after=0.0000
2026-09-26 18:14:07,865 INFO    mua.ranker.train: Targets: test ROC-AUC >= 0.700 -> FAIL (0.6312)
2026-09-26 18:14:07,865 INFO    mua.ranker.train: Targets: post-calibration test ECE <= 0.020 -> PASS (0.0047)
2026-09-26 18:14:07,865 WARNING mua.ranker.train: Diagnostics:
2026-09-26 18:14:07,865 WARNING mua.ranker.train:   - test ROC-AUC below target: the model is not separating clickers from non-clickers. Check that strong signals (merchant historical_ctr, distance vs tolerance, is_peak) are in the numeric features, try more epochs or larger embedding dims, or inspect th

In [4]:
!uv run python -m mua.cli score

2026-09-26 18:14:21,803 INFO    mua.ranker.predict: Scored train: 35006 rows, mean pctr_raw=0.42490 mean pctr=0.05941 -> /Users/abhiramkasireddi/MarketplaceUpliftAuction/data/scored/train


2026-09-26 18:14:22,001 INFO    mua.ranker.predict: Scored valid: 6664 rows, mean pctr_raw=0.42567 mean pctr=0.06002 -> /Users/abhiramkasireddi/MarketplaceUpliftAuction/data/scored/valid


2026-09-26 18:14:22,184 INFO    mua.ranker.predict: Scored test: 8330 rows, mean pctr_raw=0.42519 mean pctr=0.05991 -> /Users/abhiramkasireddi/MarketplaceUpliftAuction/data/scored/test


## 3. Causal inference

A small-scale causal config (full split sizes, lighter forest, relaxed SMD threshold).

In [5]:
from pathlib import Path

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
(root / "configs").mkdir(exist_ok=True)
(root / "configs" / "causal_small.yaml").write_text(
    """
    paths:
      processed: data/processed
      truth: data/truth
      artifacts: artifacts/causal
      figures: reports/figures
    data:
      sample_train: null
      sample_test: null
      sample_refute: 4000
      seed: 42
    propensity:
      n_estimators: 100
      trim_low: 0.02
      trim_high: 0.98
      smd_threshold: 0.12
    estimators:
      base: histgb
      bootstrap_samples: 30
      dr_cv: 2
      forest_estimators: 32
      forest_max_depth: 2
      forest_min_samples_leaf: 100
    evaluate:
      n_deciles: 10
    refute:
      placebo_iterations: 30
      subset_fractions: [0.8, 0.9]
      effect_shift_tolerance: 0.20
      placebo_pvalue_threshold: 0.05
    policy:
      ks: [10, 50, 100]
    """
)
print("wrote", root / "configs" / "causal_small.yaml")


wrote /Users/abhiramkasireddi/MarketplaceUpliftAuction/configs/causal_small.yaml


In [6]:
!uv run python -m mua.cli causal --config configs/causal_small.yaml

2026-09-26 18:14:32,503 INFO    mua.causal.data: Loaded train: 35006 rows, 35 features


2026-09-26 18:14:32,536 INFO    mua.causal.data: Loaded test: 8330 rows, 35 features + truth


2026-09-26 18:15:02,017 INFO    mua.causal.propensity: Propensity overlap plot saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/propensity_overlap.png


2026-09-26 18:15:02,442 INFO    mua.causal.propensity: Love plot saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/love_plot.png
2026-09-26 18:15:02,442 INFO    mua.causal.propensity: Propensity: AUC=0.5877 | trimmed 35006 -> 35006 rows (prop in [0.02, 0.98])
2026-09-26 18:15:02,442 INFO    mua.causal.propensity: SMD after IPTW: max=0.0448 mean=0.0109 (threshold 0.12)
2026-09-26 18:15:02,444 INFO    mua.causal.estimators: Fitting naive difference-in-means ...


2026-09-26 18:15:02,482 INFO    mua.causal.estimators: Fitting stabilized IPTW ...


...

2026-09-26 18:15:39,212 INFO    mua.causal.estimators: SHAP: skipped (requires LightGBM base models)


2026-09-26 18:15:39,301 INFO    mua.causal.evaluate: naive          ATE=0.0323 bias=+0.0088 cov=0 PEHE=0.0447 qini=0.0658 auuc=0.0183
2026-09-26 18:15:39,303 INFO    mua.causal.evaluate: iptw           ATE=0.0239 bias=+0.0004 cov=1 PEHE=0.0447 qini=0.0658 auuc=0.0183
2026-09-26 18:15:39,307 INFO    mua.causal.evaluate: s              ATE=0.0170 bias=-0.0065 cov=0 PEHE=0.0261 qini=0.3199 auuc=0.0235
2026-09-26 18:15:39,313 INFO    mua.causal.evaluate: t              ATE=0.0249 bias=+0.0013 cov=0 PEHE=0.0370 qini=0.3137 auuc=0.0239
2026-09-26 18:15:39,317 INFO    mua.causal.evaluate: x              ATE=0.0242 bias=+0.0007 cov=0 PEHE=0.0250 qini=0.3331 auuc=0.0238


2026-09-26 18:15:39,322 INFO    mua.causal.evaluate: dr             ATE=0.0231 bias=-0.0004 cov=1 PEHE=0.0290 qini=0.2934 auuc=0.0226
2026-09-26 18:15:39,327 INFO    mua.causal.evaluate: forest         ATE=0.0231 bias=-0.0004 cov=1 PEHE=0.0231 qini=0.3469 auuc=0.0238


2026-09-26 18:15:39,680 INFO    mua.causal.evaluate: Leaderboard saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/artifacts/causal/leaderboard.csv and /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/leaderboard.md


2026-09-26 18:15:39,800 INFO    mua.causal.policy: Policy (forest CATE): best-k per-dollar k=10:0.0235, k=50:0.0205, k=100:0.0114
2026-09-26 18:15:39,800 INFO    mua.causal.policy: Random targeting per-dollar: k=10:0.0185, k=50:0.0160, k=100:0.0114
2026-09-26 18:15:39,800 INFO    mua.causal.policy: Treat-everyone: incremental=96.9 orders, per-dollar=0.0041
2026-09-26 18:15:39,800 INFO    mua.causal.policy: Policy curve saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/policy_curve.png and /Users/abhiramkasireddi/MarketplaceUpliftAuction/artifacts/causal/policy_curve.csv
2026-09-26 18:15:39,800 INFO    mua.causal.run: Causal pipeline complete. Leaderboard (ranked by PEHE, then Qini):
2026-09-26 18:15:39,800 INFO    mua.causal.run:   forest         PEHE=0.0231 qini=0.3469 ATE=0.0231 bias=-0.0004
2026-09-26 18:15:39,800 INFO    mua.causal.run:   x              PEHE=0.0250 qini=0.3331 ATE=0.0242 bias=+0.0007
2026-09-26 18:15:39,800 INFO    mua.causal.run:   s       

## 4. Ads auction

In [7]:
!uv run python -m mua.cli auction --mechanism gsp --rounds 20000

2026-09-26 18:15:43,104 INFO    mua.auction.simulator: Built 645 advertisers (value mean=1.740, budget mean=0.834)


2026-09-26 18:15:44,442 INFO    mua.auction.simulator: Simulated 20000 rounds in 1.3s (15133 rounds/s)


2026-09-26 18:15:44,470 INFO    mua.auction.simulator: Auction outcomes written to /Users/abhiramkasireddi/MarketplaceUpliftAuction/data/auction_runs/gsp_s42_20260926-181544
2026-09-26 18:15:44,470 INFO    mua.auction.simulator: Platform metrics (gsp, 20000 rounds):
2026-09-26 18:15:44,470 INFO    mua.auction.simulator:   revenue=2023.12  RPM=101.1558  fill_rate=0.6005  avg_ads_position=1.525  organic_displacement=0.06021  clicks=4894
2026-09-26 18:15:44,470 INFO    mua.auction.simulator: Paced agents with >=50 wins (2): utilization p10=0.76 p50=0.77 p90=0.77 | pacing-active=2 exhausted=0
2026-09-26 18:15:44,470 INFO    mua.auction.simulator: Top spenders: adv_744(truthful)=56.59, adv_3970(random)=50.30, adv_4342(shading)=32.76, adv_3398(truthful)=29.30, adv_2646(truthful)=27.97


## 5. Experiment grid + report

In [8]:
!uv run python -m mua.cli experiment --scale small

2026-09-26 18:15:45,063 INFO    mua.reporting.experiment: Small scale: 3000 rounds x seeds [1, 2, 3]
2026-09-26 18:15:45,063 INFO    mua.reporting.experiment: Best CATE model for targeting: forest


2026-09-26 18:15:45,486 INFO    mua.causal.data: Loaded test: 8330 rows, 35 features + truth


2026-09-26 18:15:49,379 INFO    mua.reporting.experiment: Promo session pool: 8330 rows


2026-09-26 18:15:50,410 INFO    mua.auction.simulator: Built 645 advertisers (value mean=1.740, budget mean=0.219)


2026-09-26 18:15:50,837 INFO    mua.reporting.experiment: Auction cells done: first_price seed=1 revenue=696.4


...

2026-09-26 18:15:56,395 INFO    mua.reporting.experiment: Auction cells done: gsp seed=3 revenue=316.7


2026-09-26 18:15:56,901 INFO    mua.auction.simulator: Built 645 advertisers (value mean=1.740, budget mean=0.094)


2026-09-26 18:15:57,147 INFO    mua.reporting.experiment: Auction cells done: vcg seed=1 revenue=210.6


2026-09-26 18:15:57,509 INFO    mua.reporting.experiment: Auction cells done: vcg seed=2 revenue=219.9


2026-09-26 18:15:57,807 INFO    mua.reporting.experiment: Auction cells done: vcg seed=3 revenue=198.3
2026-09-26 18:15:57,827 INFO    mua.reporting.experiment: Experiment grid written to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/results.csv (60 rows)


In [9]:
!uv run python -m mua.cli report --scale small

2026-09-26 18:16:01,237 INFO    mua.auction.simulator: Built 645 advertisers (value mean=1.740, budget mean=0.063)


2026-09-26 18:16:04,226 INFO    mua.auction.simulator: Built 645 advertisers (value mean=1.740, budget mean=0.109)


2026-09-26 18:16:06,268 INFO    mua.causal.data: Loaded test: 8330 rows, 35 features + truth


2026-09-26 18:16:06,682 INFO    mua.reporting.figures: Figure saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/revenue_surplus.png
2026-09-26 18:16:06,683 INFO    mua.reporting.figures: Figure saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/frontier.png
2026-09-26 18:16:06,683 INFO    mua.reporting.figures: Figure saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/pacing_traces.png
2026-09-26 18:16:06,683 INFO    mua.reporting.figures: Figure saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/shading_convergence.png
2026-09-26 18:16:06,683 INFO    mua.reporting.figures: Figure saved to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/figures/qini_overlay.png
2026-09-26 18:16:06,690 INFO    mua.reporting.report: Report written to /Users/abhiramkasireddi/MarketplaceUpliftAuction/reports/REPORT.md


## 6. Headline numbers

In [10]:
from pathlib import Path

import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
grid = pd.read_csv(root / "reports" / "results.csv")
pivot = grid.groupby(["mechanism", "policy"]).agg(
    per_dollar_true=("per_dollar_true", "mean"),
    revenue=("revenue", "mean"),
    surplus=("advertiser_surplus", "mean"),
).round(4)
display(pivot)
print()
report = (root / "reports" / "REPORT.md").read_text()
print(report.split("## Findings")[1].split("## Limitations")[0][:1200])

per_dollar_true   revenue   surplus
mechanism    policy                                               
first_price  propensity_top_k           0.0486  702.9881  123.8228
             random_k                   0.0229  702.9881   21.5228
             treat_all                  0.0224  702.9881  207.4837
             treat_none                    NaN  702.9881  -25.5538
             uplift_top_k               0.0465  702.9881  115.2162
gsp          propensity_top_k           0.0486  334.9314  471.1012
             random_k                   0.0229  334.9314  371.4307
             treat_all                  0.0224  334.9314  553.2238
             treat_none                    NaN  334.9314  325.0216
             uplift_top_k               0.0465  334.9314  463.3405
second_price propensity_top_k           0.0486  216.9141  304.5519
             random_k                   0.0229  216.9141  240.0075
             treat_all                  0.0224  216.9141  357.1518
             treat_none                    NaN  216.9141  211.4976
             uplift_top_k               0.0465  216.9141  299.4278
vcg          propensity_top_k           0.0486  209.6090  592.0233
             random_k                   0.0229  209.6090  492.6864
             treat_all                  0.0224  209.6090  673.3169
             treat_none                    NaN  209.6090  446.5028
             uplift_top_k               0.0465  209.6090  584.0036




- **Incremental conversions per promo dollar is maximized by `propensity_top_k` targeting**: 0.0486 orders/$. For comparison: `propensity_top_k` = 0.0486, `uplift_top_k` = 0.0465, `random_k` = 0.0229, `treat_all` = 0.0224. The CATE model behind `uplift_top_k` is `forest` (leaderboard rank 1 by PEHE).
- **Revenue/surplus tradeoff:** `first_price` extracts the most platform revenue (mean 703.0), while `vcg` leaves the most advertiser surplus (mean 557.7). Advertiser welfare in first-price is driven by shading agents, not by truthful bidding.
- **Consumer welfare** (mean relevance of the shown slate) is highest under `first_price` (0.4317) and varies little across mechanisms, because all mechanisms allocate by the same eCPM ranking; the remaining differences come from bidder strategy.
- **Recommended combination:** targeting policy is the dominant lever for incremental conversions per dollar and is orthogonal to the auction mechanism; combine the best per-dollar policy above with `gsp`